In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib
import json

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from lightgbm import LGBMRegressor

pd.set_option("display.max_columns", 100)

In [2]:
PROJECT_ROOT = Path("..")

PROCESSED_DATA_PATH = PROJECT_ROOT / "01_data" / "processed" / "daily_sku_sales_processed.csv"
FEATURE_V2_PATH = PROJECT_ROOT / "01_data" / "features" / "sales_features_v2.csv"

MODEL_DIR = PROJECT_ROOT / "04_models"
MODEL_RESULTS_DIR = PROJECT_ROOT / "05_outputs" / "model_results"
FORECAST_DIR = PROJECT_ROOT / "05_outputs" / "forecasts"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FORECAST_DIR.mkdir(parents=True, exist_ok=True)

print(PROCESSED_DATA_PATH)

..\01_data\processed\daily_sku_sales_processed.csv


In [3]:
df = pd.read_csv(PROCESSED_DATA_PATH)
df["shipped_date"] = pd.to_datetime(df["shipped_date"])

df = df.sort_values(["sku", "shipped_date"]).reset_index(drop=True)

print("Shape:", df.shape)
print("Date range:", df["shipped_date"].min(), "to", df["shipped_date"].max())
print("Unique SKUs:", df["sku"].nunique())

df.head()

Shape: (29004, 7)
Date range: 2021-01-01 00:00:00 to 2022-01-02 00:00:00
Unique SKUs: 180


,shipped_date,sku,qty,revenue,cost_of_good_sold,moq_order,channel_count
0,2021-01-01,089A0E,220,5821.2,2926.0,56460,2
1,2021-01-03,089A0E,140,3704.4,2156.0,56460,1
2,2021-01-05,089A0E,150,3969.0,2310.0,56460,1
3,2021-01-09,089A0E,190,5027.4,2926.0,56460,1
4,2021-01-11,089A0E,60,1587.6,924.0,56460,1


In [4]:
# Date features
df["year"] = df["shipped_date"].dt.year
df["month"] = df["shipped_date"].dt.month
df["day"] = df["shipped_date"].dt.day
df["day_of_week"] = df["shipped_date"].dt.dayofweek
df["week_of_year"] = df["shipped_date"].dt.isocalendar().week.astype(int)
df["quarter"] = df["shipped_date"].dt.quarter
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

# SKU encoding
df["sku_encoded"] = df["sku"].astype("category").cat.codes

# Lag features
lag_periods = [1, 2, 3, 7, 14, 21, 28]

for lag in lag_periods:
    df[f"qty_lag_{lag}"] = df.groupby("sku")["qty"].shift(lag)

# Rolling features
rolling_windows = [3, 7, 14, 28]

for window in rolling_windows:
    shifted_qty = df.groupby("sku")["qty"].transform(lambda x: x.shift(1))
    
    df[f"qty_rolling_mean_{window}"] = (
        shifted_qty.groupby(df["sku"]).transform(lambda x: x.rolling(window=window).mean())
    )
    
    df[f"qty_rolling_median_{window}"] = (
        shifted_qty.groupby(df["sku"]).transform(lambda x: x.rolling(window=window).median())
    )
    
    df[f"qty_rolling_std_{window}"] = (
        shifted_qty.groupby(df["sku"]).transform(lambda x: x.rolling(window=window).std())
    )
    
    df[f"qty_rolling_min_{window}"] = (
        shifted_qty.groupby(df["sku"]).transform(lambda x: x.rolling(window=window).min())
    )
    
    df[f"qty_rolling_max_{window}"] = (
        shifted_qty.groupby(df["sku"]).transform(lambda x: x.rolling(window=window).max())
    )

# Expanding historical features
df["sku_expanding_mean_qty"] = (
    df.groupby("sku")["qty"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

df["sku_expanding_median_qty"] = (
    df.groupby("sku")["qty"]
    .transform(lambda x: x.shift(1).expanding().median())
)

df["sku_expanding_std_qty"] = (
    df.groupby("sku")["qty"]
    .transform(lambda x: x.shift(1).expanding().std())
)

# Target log
df["qty_log"] = np.log1p(df["qty"])

df.head()

,shipped_date,sku,qty,revenue,cost_of_good_sold,moq_order,channel_count,year,month,day,day_of_week,week_of_year,quarter,is_weekend,sku_encoded,qty_lag_1,qty_lag_2,qty_lag_3,qty_lag_7,qty_lag_14,qty_lag_21,qty_lag_28,qty_rolling_mean_3,qty_rolling_median_3,qty_rolling_std_3,qty_rolling_min_3,qty_rolling_max_3,qty_rolling_mean_7,qty_rolling_median_7,qty_rolling_std_7,qty_rolling_min_7,qty_rolling_max_7,qty_rolling_mean_14,qty_rolling_median_14,qty_rolling_std_14,qty_rolling_min_14,qty_rolling_max_14,qty_rolling_mean_28,qty_rolling_median_28,qty_rolling_std_28,qty_rolling_min_28,qty_rolling_max_28,sku_expanding_mean_qty,sku_expanding_median_qty,sku_expanding_std_qty,qty_log
0,2021-01-01,089A0E,220,5821.2,2926.0,56460,2,2021,1,1,4,53,1,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.398163
1,2021-01-03,089A0E,140,3704.4,2156.0,56460,1,2021,1,3,6,53,1,1,0,220.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,220.0,220.0,NaN,4.948760
2,2021-01-05,089A0E,150,3969.0,2310.0,56460,1,2021,1,5,1,1,1,0,0,140.0,220.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,180.0,180.0,56.568542,5.017280
3,2021-01-09,089A0E,190,5027.4,2926.0,56460,1,2021,1,9,5,1,1,1,0,150.0,140.0,220.0,NaN,NaN,NaN,NaN,170.0,150.0,43.588989,140.0,220.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,170.0,150.0,43.588989,5.252273
4,2021-01-11,089A0E,60,1587.6,924.0,56460,1,2021,1,11,0,2,1,0,0,190.0,150.0,140.0,NaN,NaN,NaN,NaN,160.0,150.0,26.457513,140.0,190.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,175.0,170.0,36.968455,4.110874


In [5]:
print("Before dropping missing:", df.shape)

df_features = df.dropna().reset_index(drop=True)

print("After dropping missing:", df_features.shape)
print("Unique SKUs:", df_features["sku"].nunique())
print("Date range:", df_features["shipped_date"].min(), "to", df_features["shipped_date"].max())

df_features.head()

Before dropping missing: (29004, 46)
After dropping missing: (23964, 46)
Unique SKUs: 180
Date range: 2021-02-26 00:00:00 to 2022-01-02 00:00:00


,shipped_date,sku,qty,revenue,cost_of_good_sold,moq_order,channel_count,year,month,day,day_of_week,week_of_year,quarter,is_weekend,sku_encoded,qty_lag_1,qty_lag_2,qty_lag_3,qty_lag_7,qty_lag_14,qty_lag_21,qty_lag_28,qty_rolling_mean_3,qty_rolling_median_3,qty_rolling_std_3,qty_rolling_min_3,qty_rolling_max_3,qty_rolling_mean_7,qty_rolling_median_7,qty_rolling_std_7,qty_rolling_min_7,qty_rolling_max_7,qty_rolling_mean_14,qty_rolling_median_14,qty_rolling_std_14,qty_rolling_min_14,qty_rolling_max_14,qty_rolling_mean_28,qty_rolling_median_28,qty_rolling_std_28,qty_rolling_min_28,qty_rolling_max_28,sku_expanding_mean_qty,sku_expanding_median_qty,sku_expanding_std_qty,qty_log
0,2021-03-06,089A0E,90,2381.4,1446.33,56460,2,2021,3,6,5,9,1,1,0,110.0,160.0,10.0,80.0,20.0,60.0,220.0,93.333333,110.0,76.376262,10.0,160.0,60.000000,40.0,58.878406,10.0,160.0,52.142857,40.0,45.434411,10.0,160.0,90.357143,60.0,72.952656,10.0,300.0,90.357143,60.0,72.952656,4.510860
1,2021-03-08,089A0E,70,0.0,0.00,56460,1,2021,3,8,0,10,1,0,0,90.0,110.0,160.0,10.0,60.0,190.0,140.0,120.000000,110.0,36.055513,90.0,160.0,61.428571,40.0,59.561893,10.0,160.0,57.142857,45.0,45.476718,10.0,160.0,85.714286,60.0,68.390414,10.0,300.0,90.344828,60.0,71.638116,4.262680
2,2021-03-12,089A0E,150,0.0,0.00,56460,1,2021,3,12,4,10,1,0,0,70.0,90.0,110.0,40.0,40.0,60.0,150.0,90.000000,90.0,20.000000,70.0,110.0,70.000000,70.0,55.075705,10.0,160.0,57.857143,45.0,45.603403,10.0,160.0,83.214286,60.0,67.607449,10.0,300.0,89.666667,65.0,70.490074,5.017280
3,2021-03-14,089A0E,90,2381.4,1386.00,56460,1,2021,3,14,6,10,1,1,0,150.0,70.0,90.0,10.0,100.0,40.0,190.0,103.333333,90.0,41.633320,70.0,150.0,85.714286,90.0,60.513674,10.0,160.0,65.714286,60.0,51.398037,10.0,160.0,83.214286,60.0,67.607449,10.0,300.0,91.612903,70.0,70.147310,4.510860
4,2021-03-16,089A0E,170,4498.2,2618.00,56460,1,2021,3,16,1,11,1,0,0,90.0,150.0,70.0,10.0,50.0,90.0,60.0,103.333333,90.0,41.633320,70.0,150.0,97.142857,90.0,50.568200,10.0,160.0,65.000000,60.0,50.952467,10.0,160.0,79.642857,60.0,64.318775,10.0,300.0,91.562500,75.0,69.007217,5.141664


In [6]:
df_features.to_csv(FEATURE_V2_PATH, index=False)

print(f"Saved feature v2 to: {FEATURE_V2_PATH}")
print("Shape:", df_features.shape)

Saved feature v2 to: ..\01_data\features\sales_features_v2.csv
Shape: (23964, 46)


In [7]:
exclude_cols = [
    "shipped_date",
    "sku",
    "qty",
    "qty_log",
    "revenue",
    "cost_of_good_sold",
    "channel_count"
]

feature_cols = [col for col in df_features.columns if col not in exclude_cols]

print("Number of features:", len(feature_cols))
print(feature_cols)

Number of features: 39
['moq_order', 'year', 'month', 'day', 'day_of_week', 'week_of_year', 'quarter', 'is_weekend', 'sku_encoded', 'qty_lag_1', 'qty_lag_2', 'qty_lag_3', 'qty_lag_7', 'qty_lag_14', 'qty_lag_21', 'qty_lag_28', 'qty_rolling_mean_3', 'qty_rolling_median_3', 'qty_rolling_std_3', 'qty_rolling_min_3', 'qty_rolling_max_3', 'qty_rolling_mean_7', 'qty_rolling_median_7', 'qty_rolling_std_7', 'qty_rolling_min_7', 'qty_rolling_max_7', 'qty_rolling_mean_14', 'qty_rolling_median_14', 'qty_rolling_std_14', 'qty_rolling_min_14', 'qty_rolling_max_14', 'qty_rolling_mean_28', 'qty_rolling_median_28', 'qty_rolling_std_28', 'qty_rolling_min_28', 'qty_rolling_max_28', 'sku_expanding_mean_qty', 'sku_expanding_median_qty', 'sku_expanding_std_qty']


In [8]:
unique_dates = np.sort(df_features["shipped_date"].unique())

train_end_idx = int(len(unique_dates) * 0.6)
val_end_idx = int(len(unique_dates) * 0.8)

train_end_date = unique_dates[train_end_idx]
val_end_date = unique_dates[val_end_idx]

train_df = df_features[df_features["shipped_date"] < train_end_date].copy()
val_df = df_features[
    (df_features["shipped_date"] >= train_end_date) &
    (df_features["shipped_date"] < val_end_date)
].copy()
test_df = df_features[df_features["shipped_date"] >= val_end_date].copy()

print("Train end date:", train_end_date)
print("Validation end date:", val_end_date)

print("\nTrain:", train_df.shape, train_df["shipped_date"].min(), "to", train_df["shipped_date"].max())
print("Validation:", val_df.shape, val_df["shipped_date"].min(), "to", val_df["shipped_date"].max())
print("Test:", test_df.shape, test_df["shipped_date"].min(), "to", test_df["shipped_date"].max())

Train end date: 2021-08-31T00:00:00.000000
Validation end date: 2021-11-01T00:00:00.000000

Train: (14090, 46) 2021-02-26 00:00:00 to 2021-08-29 00:00:00
Validation: (4931, 46) 2021-08-31 00:00:00 to 2021-10-30 00:00:00
Test: (4943, 46) 2021-11-01 00:00:00 to 2022-01-02 00:00:00


In [9]:
def calculate_wape(y_true, y_pred):
    denominator = np.sum(np.abs(y_true))
    
    if denominator == 0:
        return np.nan
    
    return np.sum(np.abs(y_true - y_pred)) / denominator


def evaluate_qty_predictions(y_true_qty, y_pred_qty, model_name, dataset_name):
    y_pred_qty = np.clip(y_pred_qty, 0, None)
    
    mae = mean_absolute_error(y_true_qty, y_pred_qty)
    rmse = np.sqrt(mean_squared_error(y_true_qty, y_pred_qty))
    wape = calculate_wape(y_true_qty, y_pred_qty)
    r2 = r2_score(y_true_qty, y_pred_qty)
    
    return {
        "model": model_name,
        "dataset": dataset_name,
        "MAE": mae,
        "RMSE": rmse,
        "WAPE": wape,
        "WAPE_percent": wape * 100,
        "R2": r2
    }

In [10]:
X_train = train_df[feature_cols]
X_val = val_df[feature_cols]
X_test = test_df[feature_cols]

y_train_log = train_df["qty_log"]
y_val_log = val_df["qty_log"]
y_test_log = test_df["qty_log"]

y_train_qty = train_df["qty"]
y_val_qty = val_df["qty"]
y_test_qty = test_df["qty"]

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

X_train: (14090, 39)
X_val: (4931, 39)
X_test: (4943, 39)


In [11]:
param_grid = [
    {
        "n_estimators": 500,
        "learning_rate": 0.05,
        "num_leaves": 31,
        "max_depth": -1,
        "min_child_samples": 20,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0,
        "reg_lambda": 0
    },
    {
        "n_estimators": 800,
        "learning_rate": 0.03,
        "num_leaves": 31,
        "max_depth": -1,
        "min_child_samples": 20,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0.1,
        "reg_lambda": 0.1
    },
    {
        "n_estimators": 600,
        "learning_rate": 0.04,
        "num_leaves": 63,
        "max_depth": -1,
        "min_child_samples": 30,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0.1,
        "reg_lambda": 0.3
    },
    {
        "n_estimators": 500,
        "learning_rate": 0.05,
        "num_leaves": 15,
        "max_depth": 6,
        "min_child_samples": 40,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
        "reg_alpha": 0.3,
        "reg_lambda": 0.5
    },
    {
        "n_estimators": 1000,
        "learning_rate": 0.02,
        "num_leaves": 31,
        "max_depth": 8,
        "min_child_samples": 30,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0.5,
        "reg_lambda": 0.5
    }
]

tuning_results = []
trained_models = {}

for i, params in enumerate(param_grid, start=1):
    model_name = f"LightGBM_Log_Tuned_{i}"
    
    model = LGBMRegressor(
        **params,
        objective="regression",
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )
    
    model.fit(X_train, y_train_log)
    
    val_pred_log = model.predict(X_val)
    val_pred_qty = np.expm1(val_pred_log)
    
    result = evaluate_qty_predictions(
        y_true_qty=y_val_qty,
        y_pred_qty=val_pred_qty,
        model_name=model_name,
        dataset_name="validation"
    )
    
    tuning_results.append(result)
    trained_models[model_name] = model
    
    print(model_name, "WAPE:", round(result["WAPE_percent"], 2), "%")

tuning_results_df = pd.DataFrame(tuning_results).sort_values("WAPE").reset_index(drop=True)
tuning_results_df

LightGBM_Log_Tuned_1 WAPE: 36.81 %
LightGBM_Log_Tuned_2 WAPE: 36.83 %
LightGBM_Log_Tuned_3 WAPE: 37.24 %
LightGBM_Log_Tuned_4 WAPE: 37.01 %
LightGBM_Log_Tuned_5 WAPE: 36.87 %


,model,dataset,MAE,RMSE,WAPE,WAPE_percent,R2
0,LightGBM_Log_Tuned_1,validation,123.327838,310.423341,0.368111,36.811071,0.792142
1,LightGBM_Log_Tuned_2,validation,123.386771,310.414427,0.368287,36.828661,0.792154
2,LightGBM_Log_Tuned_5,validation,123.508563,311.866623,0.368650,36.865014,0.790205
3,LightGBM_Log_Tuned_4,validation,123.989182,311.797382,0.370085,37.008470,0.790298
4,LightGBM_Log_Tuned_3,validation,124.766414,312.356487,0.372405,37.240459,0.789545


In [12]:
raw_target_results = []
raw_trained_models = {}

for i, params in enumerate(param_grid, start=1):
    model_name = f"LightGBM_Raw_Tuned_{i}"
    
    model = LGBMRegressor(
        **params,
        objective="regression",
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )
    
    model.fit(X_train, y_train_qty)
    
    val_pred_qty = model.predict(X_val)
    
    result = evaluate_qty_predictions(
        y_true_qty=y_val_qty,
        y_pred_qty=val_pred_qty,
        model_name=model_name,
        dataset_name="validation"
    )
    
    raw_target_results.append(result)
    raw_trained_models[model_name] = model
    
    print(model_name, "WAPE:", round(result["WAPE_percent"], 2), "%")

raw_target_results_df = pd.DataFrame(raw_target_results).sort_values("WAPE").reset_index(drop=True)
raw_target_results_df

LightGBM_Raw_Tuned_1 WAPE: 39.74 %
LightGBM_Raw_Tuned_2 WAPE: 39.69 %
LightGBM_Raw_Tuned_3 WAPE: 40.62 %
LightGBM_Raw_Tuned_4 WAPE: 38.15 %
LightGBM_Raw_Tuned_5 WAPE: 38.6 %


,model,dataset,MAE,RMSE,WAPE,WAPE_percent,R2
0,LightGBM_Raw_Tuned_4,validation,127.813217,312.455136,0.381499,38.149873,0.789412
1,LightGBM_Raw_Tuned_5,validation,129.314463,313.551600,0.385980,38.597968,0.787932
2,LightGBM_Raw_Tuned_2,validation,132.981622,320.372567,0.396925,39.692547,0.778605
3,LightGBM_Raw_Tuned_1,validation,133.148704,320.375501,0.397424,39.742417,0.778601
4,LightGBM_Raw_Tuned_3,validation,136.086835,321.438506,0.406194,40.619395,0.777129


In [13]:
all_tuning_results = pd.concat(
    [tuning_results_df, raw_target_results_df],
    ignore_index=True
).sort_values("WAPE").reset_index(drop=True)

all_tuning_results

,model,dataset,MAE,RMSE,WAPE,WAPE_percent,R2
0,LightGBM_Log_Tuned_1,validation,123.327838,310.423341,0.368111,36.811071,0.792142
1,LightGBM_Log_Tuned_2,validation,123.386771,310.414427,0.368287,36.828661,0.792154
2,LightGBM_Log_Tuned_5,validation,123.508563,311.866623,0.368650,36.865014,0.790205
3,LightGBM_Log_Tuned_4,validation,123.989182,311.797382,0.370085,37.008470,0.790298
4,LightGBM_Log_Tuned_3,validation,124.766414,312.356487,0.372405,37.240459,0.789545
5,LightGBM_Raw_Tuned_4,validation,127.813217,312.455136,0.381499,38.149873,0.789412
6,LightGBM_Raw_Tuned_5,validation,129.314463,313.551600,0.385980,38.597968,0.787932
7,LightGBM_Raw_Tuned_2,validation,132.981622,320.372567,0.396925,39.692547,0.778605
8,LightGBM_Raw_Tuned_1,validation,133.148704,320.375501,0.397424,39.742417,0.778601
9,LightGBM_Raw_Tuned_3,validation,136.086835,321.438506,0.406194,40.619395,0.777129


In [14]:
best_val_model_name = all_tuning_results.loc[0, "model"]
best_val_wape = all_tuning_results.loc[0, "WAPE"]

print("Best validation model:", best_val_model_name)
print("Best validation WAPE:", best_val_wape)
print("Best validation WAPE percent:", best_val_wape * 100)

if "Log" in best_val_model_name:
    best_val_model = trained_models[best_val_model_name]
    best_target_type = "log"
else:
    best_val_model = raw_trained_models[best_val_model_name]
    best_target_type = "raw"

print("Best target type:", best_target_type)

Best validation model: LightGBM_Log_Tuned_1
Best validation WAPE: 0.36811071026158143
Best validation WAPE percent: 36.811071026158146
Best target type: log


In [15]:
train_val_df = pd.concat([train_df, val_df], ignore_index=True)

X_train_val = train_val_df[feature_cols]

if best_target_type == "log":
    y_train_val = train_val_df["qty_log"]
else:
    y_train_val = train_val_df["qty"]

best_param_index = int(best_val_model_name.split("_")[-1]) - 1
best_params = param_grid[best_param_index]

final_model = LGBMRegressor(
    **best_params,
    objective="regression",
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

final_model.fit(X_train_val, y_train_val)

print("Final model trained.")
print("Best params:")
best_params

Final model trained.
Best params:


{'n_estimators': 500,
 'learning_rate': 0.05,
 'num_leaves': 31,
 'max_depth': -1,
 'min_child_samples': 20,
 'subsample': 0.8,
 'colsample_bytree': 0.8,
 'reg_alpha': 0,
 'reg_lambda': 0}

In [16]:
final_test_pred = final_model.predict(X_test)

if best_target_type == "log":
    final_test_pred_qty = np.expm1(final_test_pred)
else:
    final_test_pred_qty = final_test_pred

final_test_result = evaluate_qty_predictions(
    y_true_qty=y_test_qty,
    y_pred_qty=final_test_pred_qty,
    model_name=f"Final_{best_val_model_name}",
    dataset_name="test"
)

final_test_result

{'model': 'Final_LightGBM_Log_Tuned_1',
 'dataset': 'test',
 'MAE': 266.1869893881886,
 'RMSE': np.float64(755.7439300646002),
 'WAPE': np.float64(0.5146159292024226),
 'WAPE_percent': np.float64(51.46159292024226),
 'R2': 0.44235983757866704}

In [17]:
OLD_COMPARISON_PATH = MODEL_RESULTS_DIR / "model_comparison.csv"

if OLD_COMPARISON_PATH.exists():
    old_results = pd.read_csv(OLD_COMPARISON_PATH)
    display(old_results)
    
    old_best = old_results.sort_values("WAPE").iloc[0]
    
    comparison_summary = pd.DataFrame([
        {
            "model": old_best["model"],
            "dataset": "old_test",
            "MAE": old_best["MAE"],
            "RMSE": old_best["RMSE"],
            "WAPE": old_best["WAPE"],
            "WAPE_percent": old_best["WAPE_percent"],
            "R2": old_best["R2"]
        },
        final_test_result
    ])
else:
    comparison_summary = pd.DataFrame([final_test_result])

comparison_summary

,model,MAE,RMSE,WAPE,WAPE_percent,R2
0,LightGBM,227.563523,688.577667,0.455376,45.537580,0.510880
1,Random_Forest,228.092848,694.911941,0.456435,45.643503,0.501839
2,XGBoost,230.883111,704.849517,0.462019,46.201861,0.487489
3,Baseline_Lag_1,250.893348,821.537762,0.502061,50.206095,0.303750


,model,dataset,MAE,RMSE,WAPE,WAPE_percent,R2
0,LightGBM,old_test,227.563523,688.577667,0.455376,45.537580,0.51088
1,Final_LightGBM_Log_Tuned_1,test,266.186989,755.743930,0.514616,51.461593,0.44236


In [18]:
ATTEMPT1_TUNING_PATH = MODEL_RESULTS_DIR / "lightgbm_attempt1_validation_results.csv"
ATTEMPT1_COMPARISON_PATH = MODEL_RESULTS_DIR / "improvement_attempt1_comparison.csv"

all_tuning_results.to_csv(ATTEMPT1_TUNING_PATH, index=False)
comparison_summary.to_csv(ATTEMPT1_COMPARISON_PATH, index=False)

print(f"Saved attempt 1 tuning results to: {ATTEMPT1_TUNING_PATH}")
print(f"Saved attempt 1 comparison to: {ATTEMPT1_COMPARISON_PATH}")

Saved attempt 1 tuning results to: ..\05_outputs\model_results\lightgbm_attempt1_validation_results.csv
Saved attempt 1 comparison to: ..\05_outputs\model_results\improvement_attempt1_comparison.csv


In [19]:
FEATURE_V1_PATH = PROJECT_ROOT / "01_data" / "features" / "sales_features.csv"

df_v1 = pd.read_csv(FEATURE_V1_PATH)
df_v1["shipped_date"] = pd.to_datetime(df_v1["shipped_date"])

print("Shape:", df_v1.shape)
print("Date range:", df_v1["shipped_date"].min(), "to", df_v1["shipped_date"].max())
print("Unique SKUs:", df_v1["sku"].nunique())

df_v1.head()

Shape: (26484, 36)
Date range: 2021-01-29 00:00:00 to 2022-01-02 00:00:00
Unique SKUs: 180


,shipped_date,sku,sku_encoded,qty,qty_log,year,month,day,day_of_week,week_of_year,quarter,is_weekend,revenue,cost_of_good_sold,moq_order,channel_count,qty_lag_1,qty_lag_2,qty_lag_3,qty_lag_7,qty_lag_14,qty_rolling_mean_3,qty_rolling_std_3,qty_rolling_min_3,qty_rolling_max_3,qty_rolling_mean_7,qty_rolling_std_7,qty_rolling_min_7,qty_rolling_max_7,qty_rolling_mean_14,qty_rolling_std_14,qty_rolling_min_14,qty_rolling_max_14,sku_expanding_mean_qty,sku_expanding_median_qty,sku_expanding_std_qty
0,2021-02-04,089A0E,0,20,3.044522,2021,2,4,3,5,1,0,529.2,428.65,56460,1,300.0,110.0,90.0,60.0,220.0,166.666667,115.902258,90.0,300.0,121.428571,92.992575,40.0,300.0,128.571429,76.445772,40.0,300.0,128.571429,125.0,76.445772
1,2021-02-06,089A0E,0,60,4.110874,2021,2,6,5,5,1,1,1587.6,1285.96,56460,1,20.0,300.0,110.0,190.0,140.0,143.333333,142.945211,20.0,300.0,115.714286,98.464400,20.0,300.0,114.285714,76.732732,20.0,300.0,121.333333,110.0,78.818659
2,2021-02-10,089A0E,0,40,3.713572,2021,2,10,2,6,1,0,1058.4,857.30,56460,1,60.0,20.0,300.0,60.0,150.0,126.666667,151.437556,20.0,300.0,97.142857,94.289322,20.0,300.0,108.571429,77.643876,20.0,300.0,117.500000,100.0,77.674535
3,2021-02-12,089A0E,0,100,4.615121,2021,2,12,4,6,1,0,2646.0,2143.26,56460,1,40.0,60.0,20.0,40.0,190.0,40.000000,20.000000,20.0,60.0,94.285714,95.891804,20.0,300.0,100.714286,78.687726,20.0,300.0,112.941176,90.0,77.521344
4,2021-02-14,089A0E,0,50,3.931826,2021,2,14,6,6,1,1,1323.0,1071.63,56460,1,100.0,40.0,60.0,90.0,60.0,66.666667,30.550505,40.0,100.0,102.857143,92.864469,20.0,300.0,94.285714,74.391303,20.0,300.0,112.222222,95.0,75.268582


In [20]:
exclude_cols_v1 = [
    "shipped_date",
    "sku",
    "qty",
    "qty_log",
    "revenue",
    "cost_of_good_sold",
    "channel_count"
]

feature_cols_v1 = [col for col in df_v1.columns if col not in exclude_cols_v1]

print("Number of features:", len(feature_cols_v1))
print(feature_cols_v1)

Number of features: 29
['sku_encoded', 'year', 'month', 'day', 'day_of_week', 'week_of_year', 'quarter', 'is_weekend', 'moq_order', 'qty_lag_1', 'qty_lag_2', 'qty_lag_3', 'qty_lag_7', 'qty_lag_14', 'qty_rolling_mean_3', 'qty_rolling_std_3', 'qty_rolling_min_3', 'qty_rolling_max_3', 'qty_rolling_mean_7', 'qty_rolling_std_7', 'qty_rolling_min_7', 'qty_rolling_max_7', 'qty_rolling_mean_14', 'qty_rolling_std_14', 'qty_rolling_min_14', 'qty_rolling_max_14', 'sku_expanding_mean_qty', 'sku_expanding_median_qty', 'sku_expanding_std_qty']


In [21]:
unique_dates_v1 = np.sort(df_v1["shipped_date"].unique())

train_end_idx_v1 = int(len(unique_dates_v1) * 0.6)
val_end_idx_v1 = int(len(unique_dates_v1) * 0.8)

train_end_date_v1 = unique_dates_v1[train_end_idx_v1]
val_end_date_v1 = unique_dates_v1[val_end_idx_v1]

train_df_v1 = df_v1[df_v1["shipped_date"] < train_end_date_v1].copy()

val_df_v1 = df_v1[
    (df_v1["shipped_date"] >= train_end_date_v1) &
    (df_v1["shipped_date"] < val_end_date_v1)
].copy()

test_df_v1 = df_v1[df_v1["shipped_date"] >= val_end_date_v1].copy()

print("Train:", train_df_v1.shape, train_df_v1["shipped_date"].min(), "to", train_df_v1["shipped_date"].max())
print("Validation:", val_df_v1.shape, val_df_v1["shipped_date"].min(), "to", val_df_v1["shipped_date"].max())
print("Test:", test_df_v1.shape, test_df_v1["shipped_date"].min(), "to", test_df_v1["shipped_date"].max())

Train: (15677, 36) 2021-01-29 00:00:00 to 2021-08-17 00:00:00
Validation: (5425, 36) 2021-08-19 00:00:00 to 2021-10-24 00:00:00
Test: (5382, 36) 2021-10-26 00:00:00 to 2022-01-02 00:00:00


In [22]:
X_train_v1 = train_df_v1[feature_cols_v1]
X_val_v1 = val_df_v1[feature_cols_v1]
X_test_v1 = test_df_v1[feature_cols_v1]

y_train_log_v1 = train_df_v1["qty_log"]
y_val_log_v1 = val_df_v1["qty_log"]
y_test_qty_v1 = test_df_v1["qty"]

print("X_train:", X_train_v1.shape)
print("X_val:", X_val_v1.shape)
print("X_test:", X_test_v1.shape)

X_train: (15677, 29)
X_val: (5425, 29)
X_test: (5382, 29)


In [23]:
param_grid_v1 = [
    {
        "n_estimators": 300,
        "learning_rate": 0.05,
        "num_leaves": 31,
        "max_depth": -1,
        "min_child_samples": 20,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0,
        "reg_lambda": 0
    },
    {
        "n_estimators": 500,
        "learning_rate": 0.03,
        "num_leaves": 31,
        "max_depth": -1,
        "min_child_samples": 30,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0.1,
        "reg_lambda": 0.1
    },
    {
        "n_estimators": 400,
        "learning_rate": 0.04,
        "num_leaves": 15,
        "max_depth": 6,
        "min_child_samples": 40,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
        "reg_alpha": 0.3,
        "reg_lambda": 0.5
    },
    {
        "n_estimators": 800,
        "learning_rate": 0.02,
        "num_leaves": 31,
        "max_depth": 8,
        "min_child_samples": 30,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0.5,
        "reg_lambda": 0.5
    }
]

v1_tuning_results = []
v1_models = {}

for i, params in enumerate(param_grid_v1, start=1):
    model_name = f"LightGBM_V1_Tuned_{i}"
    
    model = LGBMRegressor(
        **params,
        objective="regression",
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )
    
    model.fit(X_train_v1, y_train_log_v1)
    
    val_pred_log = model.predict(X_val_v1)
    val_pred_qty = np.expm1(val_pred_log)
    
    result = evaluate_qty_predictions(
        y_true_qty=val_df_v1["qty"],
        y_pred_qty=val_pred_qty,
        model_name=model_name,
        dataset_name="validation"
    )
    
    v1_tuning_results.append(result)
    v1_models[model_name] = model
    
    print(model_name, "Validation WAPE:", round(result["WAPE_percent"], 2), "%")

v1_tuning_results_df = pd.DataFrame(v1_tuning_results).sort_values("WAPE").reset_index(drop=True)
v1_tuning_results_df

LightGBM_V1_Tuned_1 Validation WAPE: 34.95 %
LightGBM_V1_Tuned_2 Validation WAPE: 35.08 %
LightGBM_V1_Tuned_3 Validation WAPE: 35.02 %
LightGBM_V1_Tuned_4 Validation WAPE: 35.09 %


,model,dataset,MAE,RMSE,WAPE,WAPE_percent,R2
0,LightGBM_V1_Tuned_1,validation,117.557951,301.671386,0.349468,34.946783,0.802632
1,LightGBM_V1_Tuned_3,validation,117.813853,303.687813,0.350229,35.022856,0.799984
2,LightGBM_V1_Tuned_2,validation,117.994300,303.577936,0.350765,35.076498,0.800129
3,LightGBM_V1_Tuned_4,validation,118.056092,304.581228,0.350949,35.094867,0.798806


In [24]:
best_v1_model_name = v1_tuning_results_df.loc[0, "model"]
best_v1_param_index = int(best_v1_model_name.split("_")[-1]) - 1
best_v1_params = param_grid_v1[best_v1_param_index]

print("Best v1 validation model:", best_v1_model_name)
print("Best params:")
best_v1_params

Best v1 validation model: LightGBM_V1_Tuned_1
Best params:


{'n_estimators': 300,
 'learning_rate': 0.05,
 'num_leaves': 31,
 'max_depth': -1,
 'min_child_samples': 20,
 'subsample': 0.8,
 'colsample_bytree': 0.8,
 'reg_alpha': 0,
 'reg_lambda': 0}

In [25]:
train_val_df_v1 = pd.concat([train_df_v1, val_df_v1], ignore_index=True)

X_train_val_v1 = train_val_df_v1[feature_cols_v1]
y_train_val_log_v1 = train_val_df_v1["qty_log"]

final_v1_tuned_model = LGBMRegressor(
    **best_v1_params,
    objective="regression",
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

final_v1_tuned_model.fit(X_train_val_v1, y_train_val_log_v1)

test_pred_log_v1 = final_v1_tuned_model.predict(X_test_v1)
test_pred_qty_v1 = np.expm1(test_pred_log_v1)

final_v1_tuned_result = evaluate_qty_predictions(
    y_true_qty=y_test_qty_v1,
    y_pred_qty=test_pred_qty_v1,
    model_name=f"Final_{best_v1_model_name}",
    dataset_name="test"
)

final_v1_tuned_result

{'model': 'Final_LightGBM_V1_Tuned_1',
 'dataset': 'test',
 'MAE': 226.36216746869277,
 'RMSE': np.float64(691.4255132555961),
 'WAPE': np.float64(0.4529717777145094),
 'WAPE_percent': np.float64(45.29717777145094),
 'R2': 0.506825329921982}

In [26]:
old_best_wape = 0.455376
old_best_wape_percent = old_best_wape * 100

improvement_check = pd.DataFrame([
    {
        "model": "Old_LightGBM",
        "dataset": "old_test",
        "WAPE": old_best_wape,
        "WAPE_percent": old_best_wape_percent
    },
    {
        "model": final_v1_tuned_result["model"],
        "dataset": "test",
        "WAPE": final_v1_tuned_result["WAPE"],
        "WAPE_percent": final_v1_tuned_result["WAPE_percent"]
    }
])

improvement_check

,model,dataset,WAPE,WAPE_percent
0,Old_LightGBM,old_test,0.455376,45.537600
1,Final_LightGBM_V1_Tuned_1,test,0.452972,45.297178


In [27]:
FINAL_MODEL_PATH = MODEL_DIR / "final_sales_forecasting_model.pkl"
FINAL_FEATURE_LIST_PATH = MODEL_DIR / "final_feature_columns.json"
FINAL_MODEL_INFO_PATH = MODEL_DIR / "final_model_info.json"

joblib.dump(final_v1_tuned_model, FINAL_MODEL_PATH)

with open(FINAL_FEATURE_LIST_PATH, "w") as f:
    json.dump(feature_cols_v1, f, indent=4)

final_model_info = {
    "model_name": final_v1_tuned_result["model"],
    "base_model": "LightGBM",
    "target_type": "log",
    "trained_on_train_validation": True,
    "feature_set": "sales_features_v1",
    "train_end_date": str(train_end_date_v1),
    "validation_end_date": str(val_end_date_v1),
    "number_of_features": len(feature_cols_v1),
    "test_wape": final_v1_tuned_result["WAPE"],
    "test_wape_percent": final_v1_tuned_result["WAPE_percent"],
    "best_params": best_v1_params
}

with open(FINAL_MODEL_INFO_PATH, "w") as f:
    json.dump(final_model_info, f, indent=4)

print(f"Saved final model to: {FINAL_MODEL_PATH}")
print(f"Saved final feature columns to: {FINAL_FEATURE_LIST_PATH}")
print(f"Saved final model info to: {FINAL_MODEL_INFO_PATH}")

Saved final model to: ..\04_models\final_sales_forecasting_model.pkl
Saved final feature columns to: ..\04_models\final_feature_columns.json
Saved final model info to: ..\04_models\final_model_info.json


In [28]:
FINAL_COMPARISON_PATH = MODEL_RESULTS_DIR / "final_model_comparison.csv"

final_comparison = pd.DataFrame([
    {
        "model": "Old_LightGBM",
        "dataset": "old_test",
        "WAPE": 0.455376,
        "WAPE_percent": 45.537600
    },
    {
        "model": "Improvement_Attempt_1_Feature_V2",
        "dataset": "test",
        "WAPE": 0.514616,
        "WAPE_percent": 51.461593
    },
    {
        "model": final_v1_tuned_result["model"],
        "dataset": "test",
        "WAPE": final_v1_tuned_result["WAPE"],
        "WAPE_percent": final_v1_tuned_result["WAPE_percent"]
    }
])

final_comparison.to_csv(FINAL_COMPARISON_PATH, index=False)

print(f"Saved final comparison to: {FINAL_COMPARISON_PATH}")
final_comparison

Saved final comparison to: ..\05_outputs\model_results\final_model_comparison.csv


,model,dataset,WAPE,WAPE_percent
0,Old_LightGBM,old_test,0.455376,45.537600
1,Improvement_Attempt_1_Feature_V2,test,0.514616,51.461593
2,Final_LightGBM_V1_Tuned_1,test,0.452972,45.297178


In [29]:
final_test_forecasts = test_df_v1[["shipped_date", "sku", "qty"]].copy()

final_test_forecasts["actual_qty"] = final_test_forecasts["qty"]
final_test_forecasts["predicted_qty"] = np.clip(test_pred_qty_v1, 0, None)
final_test_forecasts["absolute_error"] = abs(
    final_test_forecasts["actual_qty"] - final_test_forecasts["predicted_qty"]
)

final_test_forecasts = final_test_forecasts.drop(columns=["qty"])

FINAL_FORECAST_PATH = FORECAST_DIR / "final_test_forecasts.csv"

final_test_forecasts.to_csv(FINAL_FORECAST_PATH, index=False)

print(f"Saved final test forecasts to: {FINAL_FORECAST_PATH}")
final_test_forecasts.head()

Saved final test forecasts to: ..\05_outputs\forecasts\final_test_forecasts.csv


,shipped_date,sku,actual_qty,predicted_qty,absolute_error
92,2021-10-26,089A0E,340,85.120237,254.879763
93,2021-10-30,089A0E,690,82.065806,607.934194
94,2021-11-01,089A0E,430,173.302765,256.697235
95,2021-11-05,089A0E,1030,294.237123,735.762877
96,2021-11-07,089A0E,1180,207.889818,972.110182


In [30]:
final_feature_importance = pd.DataFrame({
    "feature": feature_cols_v1,
    "importance": final_v1_tuned_model.feature_importances_
}).sort_values("importance", ascending=False)

FINAL_FEATURE_IMPORTANCE_PATH = MODEL_RESULTS_DIR / "final_feature_importance.csv"

final_feature_importance.to_csv(FINAL_FEATURE_IMPORTANCE_PATH, index=False)

print(f"Saved final feature importance to: {FINAL_FEATURE_IMPORTANCE_PATH}")

final_feature_importance.head(20)

Saved final feature importance to: ..\05_outputs\model_results\final_feature_importance.csv


,feature,importance
4,day_of_week,738
9,qty_lag_1,504
12,qty_lag_7,439
8,moq_order,423
19,qty_rolling_std_7,392
13,qty_lag_14,390
18,qty_rolling_mean_7,379
23,qty_rolling_std_14,373
3,day,371
15,qty_rolling_std_3,362
